In [17]:
# %%writefile ../src/CultRAG.py

"""
CultRAG.py
======================================================================
Multi-Domain RAG Orchestrator (Books + Movies + Songs)

🧠 ARCHITECTURE OVERVIEW
------------------------
This system is a 3-layer RAG pipeline:

1. INPUT LAYER
   - Normalizes memory + direct queries

2. ROUTING LAYER
   - Rewrites query for consistency (LLM-based router)
   - Routes to domain-specific RAG chains

3. DOMAIN RAG LAYER
   - BooksRAG / MoviesRAG / SongsRAG
   - Each returns structured JSON (no hallucination layer)

4. FALLBACK LAYER
   - Handles general queries outside catalog

5. PRESENTATION LAYER
   - Converts structured JSON → human-readable output

======================================================================

⚙️ DESIGN PRINCIPLES
--------------------
✔ Minimize hallucination (JSON-first design)
✔ Keep RAG outputs deterministic
✔ Keep LLM only where needed (routing + narration)
✔ Domain separation enforced at router level
✔ Memory-safe conversation handling
"""

# =========================================================
# STEP 1: ENVIRONMENT SETUP
# =========================================================
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

model = "gpt-4o-mini"

# =========================================================
# STEP 2: IMPORT DOMAIN CHAINS (RAG LAYERS)
# =========================================================
import sys, os
sys.path.append(os.path.abspath(".."))

from chain_books import chain_books
from chain_movies import chain_movies
from chain_songs import chain_songs

# =========================================================
# STEP 3: CORE LANGCHAIN IMPORTS
# =========================================================
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate
)
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# =========================================================
# STEP 4: LLM INITIALIZATION
# =========================================================
llm = ChatOpenAI(model=model, temperature=0.0)

# =========================================================
# STEP 5: INPUT NORMALIZATION LAYER
# =========================================================
def normalize_input(x):
    """
    WHY THIS EXISTS:
    ----------------
    LangChain memory returns:
    - list[BaseMessage] OR dict input

    This step ensures:
    ✔ consistent schema for downstream chains
    ✔ avoids KeyError in routing
    """

    if isinstance(x, list):
        return {
            "question": x[-1].content,
            "history": "\n".join(str(m) for m in x[:-1])
        }

    return {
        "question": x.get("question", ""),
        "history": x.get("history", "")
    }

input_mapper = RunnableLambda(normalize_input)

# =========================================================
# STEP 6: ROUTER (LLM-BASED QUERY NORMALIZER)
# =========================================================
router_prompt = PromptTemplate(
    template="""
You are a query rewriting system.

Conversation history:
{history}

User question:
{question}

TASK:
- Rewrite ONLY if needed
- Keep meaning unchanged
- Return ONLY JSON

FORMAT:
{{
  "question": "final rewritten question"
}}
""",
    input_variables=["history", "question"]
)

router_chain = router_prompt | llm | JsonOutputParser()

# =========================================================
# STEP 7: DEFAULT FALLBACK CHAIN
# =========================================================
default_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

User question:
{question}

Return ONLY valid JSON:

{
  "domain": "general",
  "query_understanding": "what user wants",
  "answer": "helpful response"
}
""")

default_chain = (
    RunnableLambda(lambda x: {"question": x["question"]})
    | default_prompt
    | llm
    | JsonOutputParser()
)

# =========================================================
# STEP 8: RULE-BASED MULTI-DOMAIN ROUTER
# =========================================================
def multi_route(x):
    """
    WHY RULE-BASED ROUTER EXISTS:
    -----------------------------
    ✔ avoids extra LLM calls
    ✔ deterministic domain routing
    ✔ faster + cheaper than semantic routing
    """

    q = x["question"].lower()
    results = []

    if "book" in q:
        results.append(chain_books.invoke(x))

    if "movie" in q:
        results.append(chain_movies.invoke(x))

    if "song" in q or "music" in q:
        results.append(chain_songs.invoke(x))

    # fallback
    if not results:
        return default_chain.invoke(x)

    # unify multi-domain outputs
    return {
        "results": results
    }

router = RunnableLambda(multi_route)

# =========================================================
# STEP 9: CORE PIPELINE (RAG ORCHESTRATION)
# =========================================================
cult_chain_core = input_mapper | router_chain | router

# =========================================================
# STEP 10: MEMORY STORE (SESSION HANDLING)
# =========================================================
store = {}

def get_session_history(session_id: str):
    """
    Simple in-memory session store.
    Replace with Redis/Postgres in production.
    """
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

cult_chain = RunnableWithMessageHistory(
    runnable=cult_chain_core,
    get_session_history=get_session_history,
    input_key="question",
    history_key="history"
)

# =========================================================
# STEP 11: NARRATION LAYER (HUMAN RESPONSE BUILDER)
# =========================================================
narrator_prompt = ChatPromptTemplate.from_template("""
You are the final presentation layer of a multi-domain RAG system.

You receive structured JSON containing results from books, movies, and songs.

TASK:
Transform the JSON into a user-facing response.

RULES:
- Preserve all factual information
- DO NOT invent new facts
- DO NOT remove domain-specific results
- DO NOT repeat identical content unnecessarily
- Keep each domain clearly separated (Books / Movies / Songs)
- Use bulleted structure
- Do not use extra formatting

OPTIONAL SYNTHESIS:
- If multiple domains are present, you MAY add 1–2 lines of cross-domain insight
- Do NOT over-analyze or hallucinate connections

FORMAT:
- Use clear sections
- Use bullets for results
- Keep it concise but complete

DO NOT add any concluding commentary or final insight outside the structured sections.
The response must end after the last listed section.


INPUT:
{data}
""")

narrator_chain = narrator_prompt | llm

def format_for_narrator(x):
    return {"data": x}

# =========================================================
# STEP 12: FINAL END-TO-END PIPELINE
# =========================================================
final_chain = (
    cult_chain
    | RunnableLambda(format_for_narrator)
    | narrator_chain
)

In [18]:
# invoke final_chain for chat type response
response = final_chain.invoke(
    "List top 2 books in adventure genre",
    config={"configurable": {"session_id": "user_1"}}
)

print(response.content)



Books:
- Title: The Winner's Crime
  - Writer: Marie Rutkoski
  - Publication Year: 2015
  - Average Rating: 4.03
  - Reading Count: 94
  - Genres: fantasy, favorites, young-adult, romance, dystopian, high-fantasy, adventure

- Title: Departure
  - Writer: A.G. Riddle
  - Publication Year: 2014
  - Average Rating: 3.58
  - Reading Count: 19
  - Genres: sci-fi, time-travel, thriller, mystery, post-apocalyptic, adventure

The results highlight two adventure genre books with varying ratings and themes.


In [15]:
# Display helpers for Jupyter
from IPython.display import display, Markdown


# Invoke cult_chain for json response
response = cult_chain.invoke(
    "List top 2 books and movies in romance genre",
    config={"configurable": {"session_id": "user_1"}}
)

print(response)

# # print(response.content)
# display(Markdown(response))

{'results': [{'domain': 'books', 'query_understanding': 'User is looking for the top 2 books in the romance genre.', 'results': [{'title': 'The Wedding (The Notebook, #2)', 'writer': 'Nicholas Sparks', 'publication_year': '2003', 'avg_rating': '3.7675765095119935', 'reading_count': '117', 'genres': 'romance, fiction, contemporary, adult, chick-lit'}, {'title': 'Wedding Night', 'writer': 'Sophie Kinsella', 'publication_year': '2013', 'avg_rating': '3.18212478920742', 'reading_count': '42', 'genres': 'chick-lit, romance, fiction, contemporary, humor'}], 'summary': "The top two romance books are 'The Wedding' by Nicholas Sparks and 'Wedding Night' by Sophie Kinsella, showcasing a mix of contemporary and chick-lit themes."}, {'domain': 'movies', 'query_understanding': 'User is looking for the top movies in the romance genre.', 'results': [{'title': 'Gay Divorcee, The', 'release_date': '01-Jan-1934', 'avg_rating': '3.8666666666666663', 'rating_count': '15', 'genres': 'Comedy, Musical, Roman